<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [98]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [99]:
type(docs)

list

In [100]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [101]:
STOPWORDS = {
    'a','an','the', 'and' , 'or' ,'but' , 'in' ,'on','at', 'to', 'for',
    'of', 'with', 'by','from', 'is', 'are','was','were', 'be'  , 'been',
    'have','has', 'had','do', 'does', 'did','will', 'would', 'could',
    'should', 'may','might', 'it', 'its' , 'this', 'that', 'these', 'those',
    'i', 'you','he','she', 'we' ,'they', 'my', 'your', 'his', 'her', 'our',
    'not' , 'no', 'so', 'if', 'as', 'up', 'out', 'about', 'into', 'than',
    'also','more', 'just'  , 'can', 'all', 'any', 'which', 'when', 'what',
    'who','how','there','their', 'them','then','than'}

In [102]:
#comlete the code
import re
from nltk.corpus import stopwords
def preprocess(text):
  text =text.lower()
  text =re.sub(r"[^a-zA-Z]"," ",text)
  tokens =text.split()
  tokens = [t for t in tokens if t not in STOPWORDS and len(t) >2]
  return tokens



![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [103]:
#comlete the code


In [104]:
def build_vocab(docs):
    all_words= set()

    for doc in docs:
        tokens = preprocess(doc)
        all_words.update(tokens)
    vocab =sorted(all_words )

    return vocab

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [105]:
def compute_df(docs, vocab):
  df = {word: 0 for word in vocab}
  for doc in docs:
        tokens =preprocess(doc)
        unique_tokens= set(tokens)
        for word in unique_tokens:
            if word in df :
                df[word]+= 1
  return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [106]:
import math

In [107]:
def compute_idf(df, N ):
  idf ={}
  for word,doc_freq in df.items():
    idf[word] = math.log((N +1)/ (doc_freq + 1))+1

  return idf


[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [113]:
def compute_tf(doc,vocab):
  tokens = preprocess(doc)
  total_words =len(tokens)
  tf_vector=np.zeros(len(vocab))

  if total_words == 0:
        return tf_vector
  word_counts = {}
  for token in tokens:
    word_counts[token] = word_counts.get(token, 0) + 1
  for i,word in enumerate(vocab):
       if word in word_counts:
           tf_vector[i] = word_counts[word] / total_words

  return tf_vector


[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [114]:
def build_tfidf(docs):
  N =len(docs)
  vocab = build_vocab(docs)
  df = compute_df(docs, vocab)
  idf= compute_idf(df,N)
  idf_vector = np.array([idf[word] for word in vocab])
  tfidf_matrix = np.zeros((N, len(vocab)))

  for i, doc in enumerate(docs):
      tf_vec = compute_tf(doc, vocab)
      tfidf_matrix[i] = tf_vec * idf_vector
  return tfidf_matrix, vocab


[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [110]:
def cosine(a ,b) :
  dot_product = np.dot(a,b)
  norm_a = np.linalg.norm(a)
  norm_b = np.linalg.norm(b)
  if norm_a == 0 or norm_b ==0 :
      return 0.0

  return dot_product / (norm_a * norm_b)


[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [111]:
import numpy as np
def search(query, docs,top_k=5):
    tfidf_matrix, vocab= build_tfidf(docs)
    q_vec = np.zeros(len(  vocab))
    q_words = preprocess(query)

    for i, w in enumerate(vocab ):
        q_vec[i]= q_words.count(w)
    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []
    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)
    return scores[:top_k]

> TEST

In [115]:
query = "machine learning neural network"
results =search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

0.21580792695913345
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.18608478341136875
OK all you experts!
Need answer quick.386 machine ,1.44 floppy ; unable to write to a formated
720 disk.Machine claims that disk is write protected,but it is not.

Note: It 'll read 720's with no prob
--------------------------------------------------
0.17552464652505897
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.15516665492150827



The previous article referred to the fact that you could only use 20ns SIMMs in
a 50MHz machine, but that you could use 80ns SIMMs in slower machines. I just
pointed out that if you could only use 
---------------------------------------------